# 그래프 품질검증 

In [31]:
# 이 셀은 전처리·적재에서도 읽는다. 데이터 읽기, API 호출, DB 변경은 하지 않는다.
# 규칙을 여기서 고치면 적재와 평가가 함께 바뀐다. 품질 정답은 아래 규칙에서 만들지 않는다.
import copy
import hashlib
import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

POLICY_VERSION = "ingredient-v2.1"
# 붙여쓰기를 허용한 재료만 등록한다. 모든 문장의 공백을 없애지는 않는다.
SPACE_NAMES = set("""
대파 쪽파 실파 청양고추 홍고추 풋고추 고춧가루 후춧가루 진간장 국간장 양조간장
맛간장 고추장 된장 쌈장 굴소스 토마토소스 올리브유 참기름 들기름 식용유
요리당 물엿 올리고당 황설탕 흑설탕 백설탕 베이킹파우더 베이킹소다
모짜렐라치즈 모차렐라치즈 파슬리가루 팽이버섯 새송이버섯 표고버섯
느타리버섯 양송이버섯 다진마늘 다진양파 다진생강 간마늘 간양파 간생강
마늘 양파 생강 감자 고구마 당근 애호박 돼지고기 소고기 닭고기
참깨 볶은깨 깨소금 소금 설탕 계란 달걀 우유 두유 멸치 다시마
""".split())
ALIASES_V2 = {"달걀": "계란", "고추가루": "고춧가루",
              "후추가루": "후춧가루", "소세지": "소시지",
              "올리브오일": "올리브유"}
SPACE_NAMES.update(ALIASES_V2)
PREP_PATTERN = re.compile(r"^(다진|간|불린|마른|볶은)\s*(마늘|양파|생강|대파|파|쌀|찹쌀|미역|당면|멸치|깨)$")
TAIL_DETAIL = re.compile(r"\s+(개인\s*입맛에.*|입맛에.*|취향에.*|생략\s*가능|초록색\s*부분.*|흰색\s*부분.*)$")
QUAL_TAIL = re.compile(r"\s*(약간|조금|적당량|취향껏|적당히|듬뿍)\s*$")
NUM_V2 = r"(?:\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)"
MEASURE_V2 = re.compile(r"(?P<a>"+NUM_V2+r")(?:\s*(?P<sep>[~\-–])\s*(?P<b>"+NUM_V2+r"))?\s*(?P<unit>큰술|작은술|티스푼|스푼|소주컵|소주잔|종이컵|센티|cm|kg|ml|mL|g|T|t|컵|개|대|장|줌|꼬집|마리|모|쪽)(?![a-zA-Z])")
UNITS_V2 = {"T":"큰술", "t":"작은술", "mL":"ml", "센티":"cm"}
# 스푼/소주컵은 용량이 확정되지 않는다. g/ml로 임의 변환하지 않는다.

def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True,
                                     separators=(",", ":")).encode()).hexdigest()

def find_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for parent in (start, *start.parents):
        if (parent / "홍기표" / "10000clean.ipynb").is_file():
            return parent
    raise FileNotFoundError("프로젝트 안에서 Notebook을 실행하세요.")

def number_v2(text):
    parts = text.strip().split()
    if len(parts) == 2:
        return float(parts[0]) + number_v2(parts[1])
    if "/" in text:
        a, b = text.split("/")
        return float(a) / float(b)
    return float(text)

def canonical_name(value):
    name = re.sub(r"\s+", " ", str(value or "")).strip()
    compact = re.sub(r"\s+", "", name)
    if compact in SPACE_NAMES:
        name = compact
    return ALIASES_V2.get(name, name)

def repair_component(item):
    # 기존 정규화가 의미를 지웠을 수 있어 name_raw부터 다시 판단한다.
    # 원래 값은 original_component에 통째로 남겨 비교/되돌리기에 사용한다.
    if item.get("normalization_version") == POLICY_VERSION:
        return copy.deepcopy(item)
    result = copy.deepcopy(item)
    original = copy.deepcopy(item)
    name = str(item.get("name_raw") or item.get("name_normalized") or "").strip()
    raw = str(item.get("raw") or name)
    details = [str(item["detail"])] if item.get("detail") else []
    prep = item.get("preparation") or []
    prep = list(prep) if isinstance(prep, list) else [str(prep)]
    reasons = []
    quantity_reasons = []
    # 이름 뒤에 남은 정성량·부연을 분리하되 원문은 변경하지 않는다.
    q = QUAL_TAIL.search(name)
    if q:
        result["qualitative_amount"] = q.group(1)
        name = name[:q.start()].strip()
    tail = TAIL_DETAIL.search(name)
    if tail:
        details.append(tail.group(1))
        name = name[:tail.start()].strip()
    measures = list(MEASURE_V2.finditer(raw))
    if measures:
        m = measures[-1]
        try:
            lo = number_v2(m["a"])
            hi = number_v2(m["b"]) if m["b"] else lo
            # 1-1/4는 혼합분수일 가능성이 있지만 출처마다 뜻이 다를 수 있다.
            # 후보 1.25는 검수용으로만 저장하고 수량 비교에서는 제외한다.
            mixed_candidate = m["sep"] == "-" and m["b"] and "/" in m["b"] and re.fullmatch(r"\d+", m["a"])
            if mixed_candidate:
                result["quantity_candidate"] = lo + hi
                quantity_reasons.append("ambiguous_mixed_fraction")
            elif lo > hi:
                quantity_reasons.append("reversed_range")
            result.update(amount_text=m.group(), unit_raw=m["unit"],
                          unit_normalized=UNITS_V2.get(m["unit"], m["unit"]),
                          amount_min=None if quantity_reasons else lo,
                          amount_max=None if quantity_reasons else hi)
        except (ValueError, ZeroDivisionError):
            quantity_reasons.append("invalid_number")
            result.update(amount_min=None, amount_max=None)
        # 이미 분리된 name_raw에는 영향이 없다. 남아 있는 수량만 걷어낸다.
        nm = MEASURE_V2.search(name)
        if nm:
            details.append(name[nm.end():].strip())
            name = name[:nm.start()].strip()
    lo, hi = result.get("amount_min"), result.get("amount_max")
    if any(x is not None and (type(x) not in (int, float) or not math.isfinite(x) or x < 0) for x in (lo, hi)):
        quantity_reasons.append("invalid_number")
    elif lo is not None and hi is not None and lo > hi:
        quantity_reasons.append("reversed_range")
    if quantity_reasons:
        result.update(amount_min=None, amount_max=None, qualitative_amount=None)
    # 명확한 준비 상태만 이동한다. 진간장/국간장, 대파/쪽파는 그대로 분리된다.
    name = canonical_name(name)
    match = PREP_PATTERN.fullmatch(name)
    if match:
        prep.append(match.group(1))
        name = canonical_name(match.group(2))
    if re.search(r"또는|혹은|(?i:or)|/", name):
        reasons.append("alternative_or_ambiguous_name")
    if re.search(r"\d|[=~()]|생략|입맛|간맞|개인|크기", name):
        reasons.append("unparsed_name_detail")
    if not name:
        reasons.append("empty_name")
    result.update(
        name_normalized=name, preparation=list(dict.fromkeys(prep)),
        detail="; ".join(x for x in details if x),
        name_review_reasons=sorted(set(reasons)),
        quantity_review_reasons=sorted(set(quantity_reasons)),
        parse_status="review" if reasons or quantity_reasons else "parsed",
        normalization_version=POLICY_VERSION,
        original_component=original,
        raw=raw,
    )
    return result

def prepare_recipe(recipe):
    result = copy.deepcopy(recipe)
    for field in ("ingredients_clean", "seasonings_clean"):
        result[field] = [repair_component(i) for i in recipe.get(field, [])]
    return result

def exclusion_reasons(recipe):
    reasons = []
    if recipe.get("graph_eligible") is not True:
        reasons.append("graph_eligible_not_true")
    for field in ("recipe_uid", "title", "source_url", "document_text", "dish_group", "dish_type"):
        if not isinstance(recipe.get(field), str) or not recipe[field].strip():
            reasons.append("missing:" + field)
    if not recipe.get("steps") or not all(isinstance(s,dict) and str(s.get("text","")).strip() for s in recipe.get("steps", [])):
        reasons.append("missing_or_invalid_steps")
    good = [i for f in ("ingredients_clean", "seasonings_clean") for i in recipe.get(f, [])
            if i.get("name_normalized") and not i.get("name_review_reasons")]
    if not good:
        reasons.append("no_usable_ingredient")
    return reasons

def select_recipes(records):
    prepared = [prepare_recipe(r) for r in records]
    accepted, excluded = [], []
    for recipe in prepared:
        reasons = exclusion_reasons(recipe)
        if reasons:
            excluded.append({"recipe_uid":recipe.get("recipe_uid"), "reasons":reasons})
        else:
            accepted.append(recipe)
    return accepted, excluded

def compare_properties(expected, actual):

    return {k: {"expected":v, "actual":actual.get(k)}
            for k,v in expected.items() if actual.get(k) != v}

def score_labels(rows, label_key, expected_count):
    complete = [r for r in rows if type(r.get(label_key)) is bool
                and str(r.get("reviewer") or "").strip() and str(r.get("reason") or "").strip()]
    status = "measured" if len(rows) == expected_count and len(complete) == expected_count else "awaiting_human_labels"
    return {"status":status, "reviewed":len(complete), "required":expected_count,
            "rate":sum(r[label_key] for r in complete)/expected_count if status == "measured" else None}


In [32]:
# 품질 정답표가 아니라 구현이 깨졌는지 확인하는 검사다.
# 기대값은 별칭표에서 생성하지 않고 개별 사례의 요구사항으로 고정했다.
assert canonical_name("대 파") == "대파"
assert canonical_name("대파") != canonical_name("쪽파")
assert canonical_name("진간장") != canonical_name("국간장")
assert canonical_name("깨소금") != canonical_name("깨")
x = repair_component({"name_raw":"다진 마늘", "raw":"다진 마늘 1T"})
assert x["name_normalized"] == "마늘" and "다진" in x["preparation"]
assert x["amount_min"] == 1 and x["unit_normalized"] == "큰술"
x = repair_component({"name_raw":"소금", "raw":"소금 1/3~1/4T"})
assert x["amount_min"] is None and x["amount_max"] is None
assert "reversed_range" in x["quantity_review_reasons"]
x = repair_component({"name_raw":"간장", "raw":"간장 1-1/4T"})
assert x["amount_min"] is None and x["quantity_candidate"] == 1.25
x = repair_component({"name_raw":"대파 초록색 부분 20센티", "raw":"대파 초록색 부분 20센티"})
assert x["name_normalized"] == "대파" and "초록색" in x["detail"]
assert repair_component(x) == x
assert repair_component({"name_raw":"고기or베이컨 생략가능", "raw":"고기or베이컨 생략가능"})["name_review_reasons"]
assert score_labels([{"correct":True,"reviewer":"tester","reason":"fixture"}], "correct", 50)["rate"] is None
assert compare_properties({"title":"A"}, {"title":"B"})["title"]["actual"] == "B"
print("회귀 검사 PASS — ER 정밀도 평가와는 별개")

# 한 건 미채점/근거 미판정이 있으면 50건 평가 완료가 될 수 없다.
labeled=[{"correct":True,"reviewer":"fixture","reason":"fixture"} for _ in range(50)]
assert score_labels(labeled,"correct",50)["rate"] == 1
labeled[-1]["correct"]=None
assert score_labels(labeled,"correct",50)["rate"] is None
assert score_labels(labeled,"evidence_supports",50)["status"]=="awaiting_human_labels"


회귀 검사 PASS — ER 정밀도 평가와는 별개


In [33]:
# 실행 설정. LLM 호출과 DB 적재는 이 Notebook에서 하지 않는다.
import os
import random
from datetime import datetime, timezone

ROOT = find_root()
WORK = ROOT / "홍기표"
INPUT = ROOT / "홍기표" / "output" / "classification_lesson_20260918_111658_927567" / "recipes_classified_cleaned.jsonl"
RUN_DB_CHECK = True       # 접속 가능하면 실제 DB를 읽어서 비교한다.
RUN_GDS = True            # DB 비교를 통과한 경우에만 임시 그래프에서 계산한다.
SEED = 42
records = [json.loads(line) for line in INPUT.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
uids = [r["recipe_uid"] for r in records]
assert len(uids) == len(set(uids)), "입력 recipe_uid 중복"
input_hash = hashlib.sha256(INPUT.read_bytes()).hexdigest()
# 검수 파일은 입력과 코드가 바뀌면 다른 폴더에 둔다. 예전 채점을 새 결과에 붙이지 않는다.
notebook_path = ROOT / "이승재" / "그래프_품질검증_v2.ipynb"
this_nb = json.loads(notebook_path.read_text(encoding="utf-8"))
code_hash = digest(["".join(c["source"]) for c in this_nb["cells"] if c["cell_type"] == "code"])
loader_path = WORK / "대분류,중분류+neo4j 적재.ipynb"
loader_nb = json.loads(loader_path.read_text(encoding="utf-8"))
helper = next("".join(c["source"]) for c in loader_nb["cells"] if "def make_graph_row" in "".join(c.get("source", [])))
run_id = digest([input_hash, code_hash, helper])[:16]
OUT = WORK / "output_quality_v2" / run_id
OUT.mkdir(parents=True, exist_ok=True)
def write_json(name, value):
    (OUT / name).write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding="utf-8")
def review_file(name, immutable_rows, editable_fields):
    # 원문·표본 ID가 수정되거나 행이 바뀌면 채점값을 읽지 않는다.
    path = OUT / name
    initial = [{"item":r, **{f:None for f in editable_fields}} for r in immutable_rows]
    if not path.exists():
        path.write_text(json.dumps(initial,ensure_ascii=False,indent=2),encoding="utf-8")
    loaded = json.loads(path.read_text(encoding="utf-8"))
    if len(loaded) != len(initial) or any(a.get("item") != b["item"] for a,b in zip(loaded,initial)):
        raise ValueError(f"{name}: 표본/근거가 바뀌었습니다. 원본 표본을 복구하세요.")
    return loaded

# 금본위 판정은 아직 비어 있다. 실제 원문 출현에서 후보를 먼저 고정한다.
mentions = []
for recipe in records:
    # 문서 단위 제외 기준부터 맞춘다. ER의 예측 결과를 보고 정답 후보를 고르지는 않는다.
    if exclusion_reasons(recipe):
        continue
    for field in ("ingredients_clean","seasonings_clean"):
        for index,item in enumerate(recipe.get(field,[])):
            mentions.append({"mention_id":f"{recipe['recipe_uid']}|{field}|{index}",
                             "recipe_uid":recipe["recipe_uid"],"field":field,"index":index,
                             "name":item.get("name_raw") or item.get("name_normalized"),
                             "raw":item.get("raw"),"source_url":recipe.get("source_url")})
by_name = {}
for m in mentions:
    if m["name"]:
        by_name.setdefault(m["name"],m)
space_groups = defaultdict(list)
for name in sorted(by_name):
    if re.fullmatch(r"[가-힣\s]+",name) and not re.search(r"또는|혹은|크기|생략|입맛|간맞|개인",name):
        space_groups[re.sub(r"\s+","",name)].append(name)
groups = [v for v in space_groups.values() if len(v)>1]
random.Random(SEED).shuffle(groups)
gold_items = []
for group in groups[:30]:
    gold_items.append({"candidate_kind":"same", "left":by_name[group[0]], "right":by_name[group[1]]})
trap_candidates = [("대파","쪽파"),("진간장","국간장"),("설탕","소금"),("참기름","들기름"),
                   ("감자","고구마"),("우유","두유"),("고추장","된장"),("돼지고기","소고기"),
                   ("양파","대파"),("굴소스","간장"),("소금","간장"),("당근","감자")]
for a,b in trap_candidates:
    if a in by_name and b in by_name and len(gold_items)<40:
        gold_items.append({"candidate_kind":"trap","left":by_name[a],"right":by_name[b]})
assert len(gold_items)==40
gold_reviews = review_file("er_gold_40.json",gold_items,["expected_same","reviewer","reason"])
# 후보 종류는 정답이 아니다. same 30 / trap 10의 판정이 실제로 확인돼야 골드가 완성된다.
print("입력:", len(records), "/ 실행 ID:", run_id)
print("검수 폴더:", OUT.relative_to(ROOT))


입력: 5000 / 실행 ID: b7ade562b0a393e2
검수 폴더: 홍기표\output_quality_v2\b7ade562b0a393e2


In [34]:
# 적재 Notebook에서 사용하는 함수 셀 자체를 호출한다.
# 연결·LLM 셀을 실행하지 않으며, payload 생성 코드의 복사본을 두지 않는다.
exec(compile(helper, str(loader_path)+":graph-functions", "exec"), globals())
accepted, excluded = select_recipes(records)
graph_rows = [make_graph_row(r) for r in accepted]
tables = split_graph_rows(graph_rows)
accepted_ids = {r["recipe_uid"] for r in accepted}
write_json("excluded_recipes.json",excluded)
write_json("graph_payload.json",tables)

occurrences = {}
component_review = []
mapping_changes = []
for recipe in records:
    prepared = prepare_recipe(recipe)
    for field in ("ingredients_clean","seasonings_clean"):
        for index,item in enumerate(prepared[field]):
            mention_id=f"{recipe['recipe_uid']}|{field}|{index}"
            occurrences[mention_id] = {"canonical_id":item["name_normalized"],
                "in_payload":recipe["recipe_uid"] in accepted_ids and not item["name_review_reasons"]}
            if item["name_review_reasons"] or item["quantity_review_reasons"]:
                component_review.append({"mention_id":mention_id,"source_url":recipe.get("source_url"),
                                         "component":item})
            before=item["original_component"].get("name_normalized")
            if before != item["name_normalized"]:
                mapping_changes.append({"mention_id":mention_id,"before":before,
                    "after":item["name_normalized"],"in_payload":occurrences[mention_id]["in_payload"],
                    "raw":item["raw"],"preparation":item["preparation"]})
write_json("component_review.json",component_review)
write_json("er_mapping_changes.json",mapping_changes)
write_json("er_policy.json",{"version":POLICY_VERSION,"space_names":sorted(SPACE_NAMES),
                            "aliases":ALIASES_V2,"code_hash":code_hash})
# name 분포는 진단용이다. 싱글톤이라는 이유만으로 오답으로 판정하지 않는다.
name_counts=Counter(e["target"] for e in tables["relations"]["INGREDIENT"])
stats={"input_count":len(records),"accepted_count":len(accepted),"excluded_count":len(excluded),
       "excluded_reasons":dict(Counter(s for r in excluded for s in r["reasons"])),
       "payload_counts":tables["expected"],"component_review_count":len(component_review),
       "changed_occurrences":len(mapping_changes),
       "unique_ingredient_count":len(name_counts),"singleton_ingredient_count":sum(n==1 for n in name_counts.values())}
print(json.dumps(stats,ensure_ascii=False,indent=2))
# 실제 적재 함수가 ER을 우회하면 깨지는 검사.
fixture = copy.deepcopy(accepted[0])
fixture["ingredients_clean"]=[{"name_raw":"고추 가루","name_normalized":"고추 가루","raw":"고추 가루 1T"}]
fixture["seasonings_clean"]=[]
edge=make_graph_row(fixture)["ingredients"][0]
assert edge["name_normalized"]=="고춧가루" and edge["source_doc_id"]==fixture["recipe_uid"]
fixture["graph_eligible"]=False
try:
    make_graph_row(fixture)
except ValueError:
    pass
else:
    raise AssertionError("적재 제외 대상이 payload에 들어갔습니다.")

# 범위 축소로 숫자가 좋아 보이지 않도록 빠진 재료 항목 수도 공개한다.
input_components=sum(len(r.get(f,[])) for r in records for f in ("ingredients_clean","seasonings_clean"))
stats["input_component_count"]=input_components
stats["payload_component_coverage"]=len(tables["relations"]["INGREDIENT"])/input_components
stats["name_review_occurrences"]=sum(bool(r["component"]["name_review_reasons"]) for r in component_review)
stats["quantity_review_occurrences"]=sum(bool(r["component"]["quantity_review_reasons"]) for r in component_review)
# 음식명 표기 후보는 사람이 확인할 목록만 만든다. 음식 종류를 임의 병합하지 않는다.
dish_groups=defaultdict(list)
for recipe in records:
    name=str(recipe.get("dish_type") or "").strip()
    if name:
        dish_groups[re.sub(r"\s+","",name)].append(
            {"name":name,"dish_group":recipe.get("dish_group"),"recipe_uid":recipe["recipe_uid"],"title":recipe.get("title")})
dish_candidates=[{"variants":sorted({r["name"] for r in group}),"examples":group[:10]}
                 for group in dish_groups.values() if len({r["name"] for r in group})>1]
write_json("dish_type_review_candidates.json",dish_candidates)


{
  "input_count": 5000,
  "accepted_count": 4742,
  "excluded_count": 258,
  "excluded_reasons": {
    "no_usable_ingredient": 229,
    "graph_eligible_not_true": 219,
    "missing:title": 7,
    "missing:document_text": 7,
    "missing_or_invalid_steps": 235,
    "missing:dish_type": 4
  },
  "payload_counts": {
    "DishGroup": 21,
    "DishType": 2365,
    "Dish": 4742,
    "Ingredient": 3367,
    "HAS_TYPE": 2365,
    "HAS_DISH": 4742,
    "INGREDIENT": 42118
  },
  "component_review_count": 4552,
  "changed_occurrences": 1581,
  "unique_ingredient_count": 3367,
  "singleton_ingredient_count": 2215
}


In [35]:
# 관계 50건을 균등 무작위 추출한다. 레시피 50개나 재료 항목 개수로 대체하지 않는다.
edges=sorted(tables["relations"]["INGREDIENT"],key=lambda e:(e["source"],e["properties"]["usage_key"]))
assert len(edges)>=50
sample=random.Random(SEED).sample(edges,50)
source_map={r["recipe_uid"]:r for r in records}
facts=[]
for edge in sample:
    recipe=source_map[edge["source"]]
    facts.append({"fact_id":edge["properties"]["usage_key"],"subject":edge["source"],
                  "relation":"INGREDIENT","object":edge["target"],
                  "properties":edge["properties"],"title":recipe["title"],
                  "document_text":recipe["document_text"],"source_url":recipe["source_url"],
                  "source_components":{"ingredients":recipe.get("ingredients_clean",[]),
                                       "seasonings":recipe.get("seasonings_clean",[])}})
fact_reviews=review_file("fact_review_50.json",facts,["correct","evidence_supports","reviewer","reason"])
precision=score_labels(fact_reviews,"correct",50)
evidence_support=score_labels(fact_reviews,"evidence_supports",50)
# 문자열 존재 여부만 자동으로 계산한다. 관계를 뒷받침하는지는 위의 사람 판정이다.
evidence_checks=[]
for edge in edges:
    evidence=edge["properties"].get("evidence") or ""
    recipe=source_map[edge["source"]]
    raw_items=[i.get("raw") for f in ("ingredients_clean","seasonings_clean") for i in recipe.get(f,[])]
    evidence_checks.append({"fact_id":edge["properties"]["usage_key"],
        "source_component_exact":bool(evidence) and evidence in raw_items,
        "document_substring":bool(evidence) and evidence in recipe.get("document_text","")})
evidence_automatic={"population":"payload INGREDIENT relationships","count":len(edges),
    "source_component_exact_rate":sum(r["source_component_exact"] for r in evidence_checks)/len(edges),
    "document_substring_rate":sum(r["document_substring"] for r in evidence_checks)/len(edges),
    "note":"원문 존재 여부만 측정. 의미적 근거 일치는 사람 채점 별도."}
write_json("evidence_checks.json",evidence_checks)

pair_results=[]
for row in gold_reviews:
    left=occurrences[row["item"]["left"]["mention_id"]]
    right=occurrences[row["item"]["right"]["mention_id"]]
    pair_results.append({"item":row["item"],"expected_same":row["expected_same"],
        "reviewer":row["reviewer"],"reason":row["reason"],
        "covered":left["in_payload"] and right["in_payload"],
        "predicted_same":left["canonical_id"]==right["canonical_id"]})
labeled=[r for r in pair_results if type(r["expected_same"]) is bool and r["reviewer"] and r["reason"]]
gold_ready=(len(labeled)==40 and sum(r["expected_same"] for r in labeled)==30)
covered=[r for r in labeled if r["covered"]]
er={"status":"awaiting_human_gold","labeled_pairs":len(labeled),"total_pairs":40,
    "covered_labeled_pairs":len(covered),"precision":None,"recall":None,"consistency":None,
    "note":"규칙표에서 만든 점수가 아님. 실제 payload에 포함된 두 출현의 ID를 비교."}
if gold_ready:
    tp=sum(r["expected_same"] and r["predicted_same"] for r in covered)
    fp=sum(not r["expected_same"] and r["predicted_same"] for r in covered)
    fn=sum(r["expected_same"] and not r["predicted_same"] for r in covered)
    er.update(status="measured" if len(covered)==40 else "incomplete_coverage",
        precision=tp/(tp+fp) if tp+fp else None, recall=tp/(tp+fn) if tp+fn else None,
        consistency=sum(r["expected_same"]==r["predicted_same"] for r in covered)/len(covered) if covered else None,
        false_merge_count=fp, missed_merge_count=fn)
write_json("er_pair_results.json",pair_results)

# 입력 구조와 추출 스키마는 분모가 다르다.
input_valid=sum(all(isinstance(r.get(k),str) and r[k].strip() for k in ("recipe_uid","title","source_url","document_text"))
                and all(isinstance(r.get(k),list) for k in ("ingredients_clean","seasonings_clean","steps")) for r in records)
input_structure={"valid":input_valid,"total":len(records),"rate":input_valid/len(records),
                 "definition":"문서 필수 문자열 및 목록 타입 검사. LLM 추출 준수율 아님."}
extraction_compliance={"status":"unavailable","rate":None,
    "reason":"이 입력은 정형 재료 및 음식 분류 결과다. 비정형 트리플 후보/승인/거절 로그가 연결되지 않았다."}
# 실제 graph payload의 구조도 별도 검사한다.
payload_issues=[]
node_keys={"DishGroup":"name","DishType":"key","Dish":"recipe_uid","Ingredient":"name_normalized"}
for label,key in node_keys.items():
    keys=[n.get(key) for n in tables["nodes"][label]]
    if not all(keys) or len(keys)!=len(set(keys)):
        payload_issues.append("invalid_or_duplicate_key:"+label)
for edge in edges:
    p=edge["properties"]
    if edge["source"] not in accepted_ids or edge["target"] not in name_counts:
        payload_issues.append("missing_endpoint:"+p["usage_key"])
    if p.get("amount_min") is not None and p.get("amount_max") is not None and p["amount_min"]>p["amount_max"]:
        payload_issues.append("reversed_quantity:"+p["usage_key"])
payload_validation={"status":"passed" if not payload_issues else "failed","issues":payload_issues}
print("50건 정밀도:",precision)
print("ER 골드 평가:",er)
print("LLM 추출 준수율:",extraction_compliance["status"])


50건 정밀도: {'status': 'awaiting_human_labels', 'reviewed': 0, 'required': 50, 'rate': None}
ER 골드 평가: {'status': 'awaiting_human_gold', 'labeled_pairs': 0, 'total_pairs': 40, 'covered_labeled_pairs': 0, 'precision': None, 'recall': None, 'consistency': None, 'note': '규칙표에서 만든 점수가 아님. 실제 payload에 포함된 두 출현의 ID를 비교.'}
LLM 추출 준수율: unavailable


In [36]:
# DB는 읽어서만 비교한다. 접속 실패를 성공/건너뜀으로 바꾸지 않는다.
def inspect_database(tables):
    if not RUN_DB_CHECK:
        return {"status":"not_run","reason":"RUN_DB_CHECK=False"}
    try:
        from dotenv import load_dotenv
        from neo4j import GraphDatabase
        for env in (WORK / ".env", ROOT / ".env"):
            if env.is_file():
                load_dotenv(env,override=False)
        uri=os.getenv("NEO4J_URI")
        user=os.getenv("NEO4J_USER")
        password=os.getenv("NEO4J_PASSWORD")
        if not all((uri,user,password)):
            return {"status":"unavailable","reason":"Neo4j 접속 환경변수 누락"}
        issues=[]
        counts={}
        # 이 검증은 전용 레시피 DB를 기준으로 한다. 기존 데이터가 섞이면 차이를 보고한다.
        with GraphDatabase.driver(uri,auth=(user,password),connection_timeout=5,
                                  connection_acquisition_timeout=8) as driver:
            driver.verify_connectivity()
            with driver.session(database=os.getenv("NEO4J_DATABASE","neo4j")) as session:
                for label,key in node_keys.items():
                    actual=[dict(r["p"]) for r in session.run(f"MATCH (n:{label}) RETURN properties(n) AS p LIMIT 100001")]
                    if len(actual)>100000:
                        return {"status":"failed","reason":"검증 범위 100000노드 초과"}
                    expected_nodes=tables["nodes"][label]
                    counts[label]={"expected":len(expected_nodes),"actual":len(actual)}
                    actual_by_key=defaultdict(list)
                    for n in actual:
                        actual_by_key[n.get(key)].append(n)
                    expected_keys={n[key] for n in expected_nodes}
                    for n in expected_nodes:
                        matches=actual_by_key.get(n[key],[])
                        if len(matches)!=1:
                            issues.append({"label":label,"key":n[key],"match_count":len(matches)})
                        else:
                            diff=compare_properties(n,matches[0])
                            if diff: issues.append({"label":label,"key":n[key],"property_diff":diff})
                    for k in set(actual_by_key)-expected_keys:
                        issues.append({"label":label,"extra_key":k})
                specs={"HAS_TYPE":("DishGroup","name","DishType","key"),
                       "HAS_DISH":("DishType","key","Dish","recipe_uid"),
                       "INGREDIENT":("Dish","recipe_uid","Ingredient","name_normalized")}
                for kind,(sl,sk,tl,tk) in specs.items():
                    actual=[r.data() for r in session.run(
                        f"MATCH (a:{sl})-[r:{kind}]->(b:{tl}) RETURN a.{sk} AS source,b.{tk} AS target,properties(r) AS properties LIMIT 100001")]
                    if len(actual)>100000:
                        return {"status":"failed","reason":"검증 범위 100000관계 초과"}
                    expected_edges=tables["relations"][kind]
                    counts[kind]={"expected":len(expected_edges),"actual":len(actual)}
                    def edge_key(e):
                        return (e["source"],e["target"],e.get("properties",{}).get("usage_key"))
                    actual_map=defaultdict(list)
                    for e in actual: actual_map[edge_key(e)].append(e)
                    expected_keys={edge_key(e) for e in expected_edges}
                    for e in expected_edges:
                        matches=actual_map.get(edge_key(e),[])
                        if len(matches)!=1:
                            issues.append({"relation":kind,"key":edge_key(e),"match_count":len(matches)})
                        else:
                            diff=compare_properties(e.get("properties",{}),matches[0]["properties"])
                            if diff: issues.append({"relation":kind,"key":edge_key(e),"property_diff":diff})
                    for k in set(actual_map)-expected_keys:
                        issues.append({"relation":kind,"extra_key":k})
                constraints=[r.data() for r in session.run("SHOW CONSTRAINTS YIELD type, labelsOrTypes, properties RETURN type, labelsOrTypes, properties")]
                for label,key in node_keys.items():
                    if not any(c["labelsOrTypes"]==[label] and c["properties"]==[key] and c["type"] in ("UNIQUENESS","NODE_KEY","NODE_PROPERTY_UNIQUENESS") for c in constraints):
                        issues.append({"missing_unique_constraint":f"{label}.{key}"})
        write_json("db_differences.json",issues)
        return {"status":"passed" if not issues else "failed","counts":counts,
                "difference_count":len(issues),"examples":issues[:10]}
    except Exception as error:
        # URI에 인증정보가 포함될 수도 있으므로 예외 전문을 보고서에 저장하지 않는다.
        return {"status":"unavailable","error_type":type(error).__name__,
                "reason":"접속/조회 실패. Neo4j 실행·계정·DB 이름을 로컬에서 확인하세요."}

db_report=inspect_database(tables)
print("DB 비교:",db_report["status"],db_report.get("reason",""))


DB 비교: failed 


In [37]:
# GDS는 DB 결과가 payload와 일치할 때만 수행한다.
# 역할별 재료 사용은 보존하되 분석 그래프에는 Dish/Ingredient 쌍을 한 번만 넣는다.
def gds_analysis():
    if not RUN_GDS:
        return {"status":"not_run","reason":"RUN_GDS=False"}
    if db_report["status"]!="passed":
        return {"status":"blocked","reason":"DB/payload 비교 미통과"}
    from neo4j import GraphDatabase
    from uuid import uuid4
    graph_name="quality_v2_"+uuid4().hex
    community_name=graph_name+"_community"
    names=[]
    try:
        with GraphDatabase.driver(os.environ["NEO4J_URI"],auth=(os.environ["NEO4J_USER"],os.environ["NEO4J_PASSWORD"]),
                                  connection_timeout=5) as driver:
            with driver.session(database=os.getenv("NEO4J_DATABASE","neo4j")) as session:
                try:
                    version=session.run("RETURN gds.version() AS v").single()["v"]
                    session.run("""
                    MATCH (d:Dish)-[:INGREDIENT]->(i:Ingredient)
                    WITH DISTINCT d,i
                    WITH gds.graph.project($name,d,i,
                      {}, {undirectedRelationshipTypes:['*']}) AS g
                    RETURN g.nodeCount AS nodes,g.relationshipCount AS relationships
                    """,name=graph_name).consume()
                    names.append(graph_name)
                    top=[r.data() for r in session.run("""
                    CALL gds.pageRank.stream($name) YIELD nodeId,score
                    WITH gds.util.asNode(nodeId) AS n,score WHERE n:Ingredient
                    RETURN n.name_normalized AS ingredient,score
                    ORDER BY score DESC,ingredient LIMIT 10
                    """,name=graph_name)]
                    # 소금·물 같은 흔한 재료 두 개만으로 모든 요리를 연결하지 않는다.
                    # 15% 이상 레시피에 나오는 재료를 유사도 계산에서 제외한다.
                    pair_count=session.run("""
                    MATCH (i:Ingredient)<-[:INGREDIENT]-(d:Dish)
                    WITH i,count(DISTINCT d) AS df WHERE df < $max_df
                    MATCH (a:Dish)-[:INGREDIENT]->(i)<-[:INGREDIENT]-(b:Dish)
                    WHERE a.recipe_uid < b.recipe_uid
                    WITH a,b,count(DISTINCT i) AS shared WHERE shared >= 2
                    MATCH (a)-[:INGREDIENT]->(ai:Ingredient)
                    WITH a,b,shared,count(DISTINCT ai) AS da
                    MATCH (b)-[:INGREDIENT]->(bi:Ingredient)
                    WITH a,b,shared,da,count(DISTINCT bi) AS db
                    WITH a,b,1.0*shared/(da+db-shared) AS weight
                    ORDER BY weight DESC,a.recipe_uid,b.recipe_uid LIMIT 50000
                    WITH gds.graph.project($name,a,b,
                      {relationshipProperties:{weight:weight}},
                      {undirectedRelationshipTypes:['*']}) AS g
                    RETURN g.nodeCount AS nodes,g.relationshipCount AS relationships
                    """,name=community_name,max_df=max(2,len(accepted)*0.15)).single()
                    if pair_count is None:
                        return {"status":"incomplete","pagerank_top10":top,"communities_top5":[],
                                "reason":"유사도 조건을 만족하는 레시피 쌍 없음"}
                    names.append(community_name)
                    communities=[r.data() for r in session.run("""
                    CALL gds.louvain.stream($name,{relationshipWeightProperty:'weight',concurrency:1})
                    YIELD nodeId,communityId
                    WITH communityId,gds.util.asNode(nodeId) AS d
                    ORDER BY d.recipe_uid
                    WITH communityId,count(*) AS recipe_count,collect({uid:d.recipe_uid,title:d.title})[..5] AS examples
                    RETURN communityId AS community_id,recipe_count,examples
                    ORDER BY recipe_count DESC,community_id LIMIT 5
                    """,name=community_name)]
                    return {"status":"measured" if len(top)==10 and len(communities)==5 else "incomplete",
                            "gds_version":version,"pagerank_top10":top,"communities_top5":communities,
                            "similarity_projection":dict(pair_count),
                            "projection_policy":{"deduplicate_usage":True,"max_df_fraction":0.15,
                                                 "min_shared":2,"edge_cap":50000,
                                                 "weight":"shared_infrequent / (full_degree_a + full_degree_b - shared_infrequent)"},
                            "note":"상위 5개 커뮤니티이며 알고리즘이 정확히 5개로 나누도록 강제하지 않는다."}
                finally:
                    for name in reversed(names):
                        session.run("CALL gds.graph.drop($name,false) YIELD graphName RETURN graphName",name=name).consume()
    except Exception as error:
        return {"status":"failed","error_type":type(error).__name__,
                "reason":"GDS 버전/프로시저/메모리를 확인하세요. 자동으로 통과 처리하지 않습니다."}

gds_report=gds_analysis()
print("GDS:",gds_report["status"])


GDS: blocked


In [38]:
# 자동 검사 PASS와 프로젝트 평가 완료를 구분한다.
checks={"payload":payload_validation["status"],"db":db_report["status"],
        "sample_precision":precision["status"],"evidence_support":evidence_support["status"],
        "er_gold":er["status"],"llm_extraction_compliance":extraction_compliance["status"],"gds":gds_report["status"]}
report={"version":"2.1","status":"complete" if all(v in ("passed","measured") for v in checks.values()) else "incomplete",
        "generated_at_utc":datetime.now(timezone.utc).isoformat(),"checks":checks,
        "provenance":{"input":str(INPUT.relative_to(ROOT)),"input_sha256":input_hash,
                      "policy_version":POLICY_VERSION,"code_hash":code_hash,"run_id":run_id,
                      "evaluation_scope":"current Hong Dish/Ingredient payload; not the earlier Recipe ontology"},
        "stats":stats,"input_structure":input_structure,"payload_validation":payload_validation,
        "llm_extraction_compliance":extraction_compliance,"sample_precision":precision,
        "evidence_automatic":evidence_automatic,"evidence_support":evidence_support,
        "er":er,"db":db_report,"gds":gds_report,
        "remaining":["er_gold_40.json의 expected_same/reviewer/reason 검수",
                     "fact_review_50.json의 correct/evidence_supports/reviewer/reason 검수",
                     "component_review.json 및 dish_type_review_candidates.json 검토",
                     "실제 비정형 추출 후보·거절 로그 연결",
                     "DB 차이가 있으면 별도 검증 DB 적재 후 다시 비교"]}
write_json("quality_report_v2.json",report)
print(json.dumps({"status":report["status"],"checks":checks,"report":str((OUT/"quality_report_v2.json").relative_to(ROOT))},ensure_ascii=False,indent=2))


{
  "status": "incomplete",
  "checks": {
    "payload": "passed",
    "db": "failed",
    "sample_precision": "awaiting_human_labels",
    "evidence_support": "awaiting_human_labels",
    "er_gold": "awaiting_human_gold",
    "llm_extraction_compliance": "unavailable",
    "gds": "blocked"
  },
  "report": "홍기표\\output_quality_v2\\b7ade562b0a393e2\\quality_report_v2.json"
}


## 검수 및 실행 순서

1. Run All은 유료 API를 호출하지 않는다. 새 결과 폴더에 JSON을 저장하고 DB는 읽어서 비교한다.
2. `er_gold_40.json`: 후보 종류를 정답으로 믿지 말고 두 원문을 보고 `expected_same`에 true/false, `reviewer`, `reason`을 입력한다. 30 동일/10 상이 판정이 충족되지 않으면 완료되지 않는다.
3. `fact_review_50.json`: 관계와 속성이 정확하면 `correct=true`, 근거가 관계를 지지하면 `evidence_supports=true`. 틀리면 false와 이유를 입력한다. `item` 내용은 수정하지 않는다.
4. 같은 입력/코드로 재실행하면 채점을 재사용한다. 코드나 입력이 바뀌면 새 폴더를 사용한다.
5. 기존 DB와 차이가 있으면 `db_differences.json`을 먼저 확인한다. 이 Notebook은 기존 데이터를 삭제하지 않는다.
6. 레시피 목록/근거 텍스트는 payload에 보존하지만 벡터·full-text 인덱스 및 검색 비교는 후속 작업이다.
